# 📖 Bible Reading Progress Tracker — NER BERT Fine-tuning

**Purpose**  
Fine-tune IndoBERT for Named Entity Recognition (NER) on Bible reading progress messages.

| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup** | Experiment config, imports |
| 2 | **Model & Tokenizer** | Load IndoBERT base + label schema |
| 3 | **Data** | Load CoNLL, split, tokenize |
| 4 | **Training** | Fine-tune with `Trainer` |
| 5 | **Evaluation** | Metrics + save model |
| 6 | **Inference** | Sanity-check pipeline on test sentences |

## 1. Setup

### 1.1 Experiment Config

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve() / 'src'))

from config.settings import PROCESSED_DIR, CONFIG_PATH, MODEL_DIR

# ── Experiment Config ──────────────────────────────────────────────────────
EXPERIMENT_NAME = "indobert-bible-ner-v4"
DATA_PATH  = PROCESSED_DIR / "NER_tasks" / "ner_tasks_3000.conll"
SAVE_PATH = MODEL_DIR / EXPERIMENT_NAME
OUTPUT_DIR = f'./runs/{EXPERIMENT_NAME}'

# ── Training knobs ───────────────────────────────────────────────────────────
NUM_EPOCHS = 30 

print(f'Experiment      : {EXPERIMENT_NAME}')
print(f'Data            : {DATA_PATH}')
print(f'Max epochs      : {NUM_EPOCHS}')
print(f'Save path       : {SAVE_PATH}')

Experiment      : indobert-bible-ner-v4
Data            : C:\one one\Desktop\bible_reading_recap_nlp\data\processed\NER_tasks\ner_tasks_3000.conll
Max epochs      : 30
Save path       : C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v4


### 1.2 Imports

In [2]:
import re
import ast
import emoji
import unicodedata
import warnings
import numpy as np
from typing import Any

import evaluate
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    EarlyStoppingCallback,
    DataCollatorForTokenClassification,
    TrainingArguments, 
    Trainer,
)

warnings.filterwarnings('ignore')

from bible_data import load_bible_data
from extraction.extractor import load_ner_model

print('Setup complete.')

Setup complete.


## 2. Model & Tokenizer

**Base model:** IndoBERT — pre-trained on Indonesian text, which gives it implicit knowledge of Indonesian Bible book names even before fine-tuning.

**Label schema** — BIO tagging so `seqeval` evaluates entity boundaries correctly:

| Label | Meaning |
|-------|---------|
| `O` | Outside any entity |
| `B-BOOK` | First token of a book name |
| `I-BOOK` | Continuation token of a book name |
| `B-CHAPTER` | First token of a chapter number |
| `I-CHAPTER` | Continuation token of a chapter number |

In [3]:
model, tokenizer = load_ner_model(CONFIG_PATH)

bible_data = load_bible_data()

ORIGINAL_VOCAB_SIZE = len(tokenizer.get_vocab())

# Compound book names that WordPiece splits incorrectly
compound_tokens = [
    book['name']
    for book in bible_data['books'] if '-' in book['name']
]

existing = set(tokenizer.get_vocab().keys())
new_tokens = [t for t in compound_tokens if t not in existing]

tokenizer.add_tokens(new_tokens)
print(f'Added {len(new_tokens)} tokens: {new_tokens}')
print(f'Vocab size: {len(tokenizer)}')

2026-04-11 14:29:49 | INFO     | bible_pipeline.extraction.extractor | Loading pretrained model: indolem/indobert-base-uncased


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

Added 3 tokens: ['Hakim-hakim', '1 Raja-raja', '2 Raja-raja']
Vocab size: 31926


In [4]:
# Sanity-check: verify compound tokens are now single ids
for tok in new_tokens:
    ids = tokenizer.convert_tokens_to_ids([tok])
    pieces = tokenizer.tokenize(tok)
    status = '✓' if len(pieces) == 1 else '✗ still splits'
    print(f'  {tok!r:20s} → id={ids[0]}  pieces={pieces}  {status}')

  'Hakim-hakim'        → id=31923  pieces=['hakim-hakim']  ✓
  '1 Raja-raja'        → id=31924  pieces=['1 raja-raja']  ✓
  '2 Raja-raja'        → id=31925  pieces=['2 raja-raja']  ✓


In [5]:
model.resize_token_embeddings(len(tokenizer))

id2label =  model.config.id2label
label2id = model.config.label2id
print("Label schema", id2label)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Label schema {0: 'O', 1: 'B-BIBLE_REF', 2: 'I-BIBLE_REF'}


In [6]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Trainable params : {trainable_params:,}')

Trainable params : 109,972,227


## 3. Data

### 3.1 Load CoNLL

CoNLL format - one token per line, blank lines seperate sentences.

```
Efesus  B-BIBLE_REF
1       I-BIBLE_REF
-       I-BIBLE_REF
2       I-BIBLE_REF
done    O
```

In [7]:
def read_conll(filepath: Path):
    """
    Parse a CoNLL-format file into (sentences, labels) lists.
    """
    sentences, labels, raw_texts = [], [], []
    tokens, ner_tags = [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)
                    raw_texts.append(' '.join(tokens))
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if parts[0] == "-DOCSTART-":
                    continue

                tokens.append(parts[0])
                ner_tags.append(parts[-1])
    if tokens:
        sentences.append(tokens)
        labels.append(ner_tags)
        raw_texts.append(' '.join(tokens))

    return sentences, labels, raw_texts

In [8]:
CROSS_CHAPTER_VERSE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+:\d+',
    re.IGNORECASE,
)
VERSE_RANGE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+(?!\s*:)'
    r'|\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+(?!\s*:)',
    re.IGNORECASE,
)
SINGLE_VERSE_RE = re.compile(r'\d+:\d+')
CROSS_BOOK_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*[A-Za-z]'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+[A-Za-z]',
    re.IGNORECASE,
)
CHAPTER_RANGE_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*\d+'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+\d+',
    re.IGNORECASE,
)
MARKDOWN_RE = re.compile(r'[_*]')


def is_noisy(msg: str) -> bool:
    has_emoji = any(unicodedata.category(c) in ('So', 'Sm') or c in emoji.EMOJI_DATA for c in msg)
    has_markdown = bool(MARKDOWN_RE.search(msg))
    return has_emoji or has_markdown


def parse_spans(spans: Any) -> list[dict]:
    if isinstance(spans, str):
        return ast.literal_eval(spans)
    return spans or []

def detect_pattern_from_conll(tokens: list[str], ner_tags: list[str]) -> str:
    """Infer pattern pool from token list + BIO labels."""
    text     = ' '.join(tokens)
    ref_toks = [t for t, l in zip(tokens, ner_tags) if l != 'O']
    ref_text = ' '.join(ref_toks)

    if is_noisy(text):
        return 'noisy'

    if not ref_toks:
        return 'none'

    # Count distinct B- spans → multi if ≥ 2
    b_count = sum(1 for l in ner_tags if l == 'B-BIBLE_REF')
    if b_count >= 2:
        return 'multi'

    # Classify the single span by its text
    if CROSS_CHAPTER_VERSE_RE.search(ref_text):
        return 'cross_chapter_verse'
    if VERSE_RANGE_RE.search(ref_text):
        return 'verse_range'
    if SINGLE_VERSE_RE.search(ref_text):
        return 'single_verse'
    if CROSS_BOOK_RE.search(ref_text):
        return 'cross_book'
    if CHAPTER_RANGE_RE.search(ref_text):
        return 'chapter_range'
    if re.search(r'\d', ref_text):
        return 'single_chapter'
    return 'book_only'

In [9]:
sentences, labels, raw_texts = read_conll(DATA_PATH)

pattern_labels = [
    detect_pattern_from_conll(toks, tags)
    for toks, tags in zip(sentences, labels)
]

from collections import Counter
print("Pattern distribution in dataset:")
for pat, n in sorted(Counter(pattern_labels).items()):
    print(f"  {pat:22s}: {n}")

Pattern distribution in dataset:
  book_only             : 128
  chapter_range         : 740
  cross_book            : 444
  cross_chapter_verse   : 17
  multi                 : 639
  noisy                 : 678
  none                  : 106
  single_chapter        : 196
  single_verse          : 9
  verse_range           : 43


### 3.2 Train / Eval Split

80/20 stratified by label presence — ensures both splits see `B-BOOK`, `I-BOOK`, `B-CHAPTER`, `I-CHAPTER` examples.

In [10]:
(train_sent, eval_sent,
 train_labels, eval_labels,
 train_patterns, eval_patterns) = train_test_split(
    sentences, labels, pattern_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=pattern_labels,
)

print(f'Train : {len(train_sent)} | Eval : {len(eval_sent)}')
print(f'\nEval pattern breakdown:')
for pat, n in sorted(Counter(eval_patterns).items()):
    print(f'  {pat:22s}: {n}')

Train : 2400 | Eval : 600

Eval pattern breakdown:
  book_only             : 26
  chapter_range         : 148
  cross_book            : 89
  cross_chapter_verse   : 3
  multi                 : 128
  noisy                 : 136
  none                  : 21
  single_chapter        : 39
  single_verse          : 2
  verse_range           : 8


### 3.3 Tokenize & Align Labels

IndoBERT uses WordPiece tokenization — a single word like `Korintus` may split into multiple subword tokens.  
Only the **first subword** of each word gets the real label; continuation subwords are masked with `-100` so they are ignored by the loss.

In [11]:
raw_dataset = DatasetDict({
    'train': Dataset.from_dict({
        'tokens': train_sent,
        'ner_tags': train_labels,
        'pattern': train_patterns,
    }),
    'eval': Dataset.from_dict({
        'tokens': eval_sent,
        'ner_tags': eval_labels,
        'pattern': eval_patterns,
    }),
})

In [12]:
def tokenize_and_align_labels(example):
    """
    Tokenise a pre-split sentence and propagate NER labels to subword tokens.
    Continuation tokens (word_idx == previous_word_idx) are masked with -100
    so they are ignored by the loss.
    """
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[example["ner_tags"][word_idx]])
        else:
            label_ids.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [13]:
tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_dataset = tokenized_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_dataset.set_format("torch")
tokenized_dataset

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "C:\Python312\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Python312\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\transformers\safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "c:\one one\Desktop\bible_readi

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pattern', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2400
    })
    eval: Dataset({
        features: ['pattern', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 600
    })
})

## 4. Training

### 4.1 Metrics
Using `seqeval` for entity-level evaluation.

In [14]:
def make_compute_metrics(id2label, eval_patterns):
    seqeval = evaluate.load('seqeval')

    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions = np.argmax(logits, axis=-1)

        true_labels, true_preds = [], []
        for pred_seq, label_seq in zip(predictions, labels):
            cur_l, cur_p = [], []
            for pred_id, label_id in zip(pred_seq, label_seq):
                if label_id != -100:
                    cur_l.append(id2label[label_id])
                    cur_p.append(id2label[pred_id])
            true_labels.append(cur_l)
            true_preds.append(cur_p)

        overall = seqeval.compute(
            predictions=true_preds, 
            references=true_labels, 
            mode='strict'
        )

        per_pattern = {}
        for pat in set(eval_patterns):
            idx = [i for i, p in enumerate(eval_patterns) if p == pat]
            if not idx:
                continue

            pat_true = [true_labels[i] for i in idx]
            pat_pred = [true_preds[i] for i in idx]

            has_entities = any(
                l != 'O' for seq in pat_true for l in seq
            )
            if not has_entities:
                per_pattern[f'f1_{pat}'] = 0.0
                continue

            r = seqeval.compute(
                predictions=pat_pred,
                references=pat_true,
                mode='strict',
            )
            per_pattern[f'f1_{pat}'] = round(r['overall_f1'], 4)

        return {
            'precision': overall['overall_precision'],
            'recall': overall['overall_recall'],
            'f1': overall['overall_f1'],
            'token_accuracy': overall['overall_accuracy'],
            **per_pattern,
        }

    return compute_metrics

### 4.2 Training Arguments

In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=42,

    # Optimiser
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=int(0.1 * (len(train_sent) / (8 * 2)) * NUM_EPOCHS),

    # Batch / accumulation
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    # Schedule
    lr_scheduler_type="cosine",
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=1.0,

    # Checkpoint
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=3,
    logging_steps=10,
)

### 4.3 Train

In [16]:
compute_metrics = make_compute_metrics(id2label, eval_patterns)

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=5,
        early_stopping_threshold=0.001
        )]
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy,F1 None,F1 Single Chapter,F1 Noisy,F1 Verse Range,F1 Book Only,F1 Chapter Range,F1 Cross Book,F1 Multi,F1 Cross Chapter Verse,F1 Single Verse
1,0.450269,0.111223,0.917303,0.943717,0.930323,0.971129,0.000000,0.861100,0.929000,1.000000,0.943400,0.963000,0.944400,0.934900,1.000000,1.000000
2,0.078516,0.024964,0.984395,0.990838,0.987606,0.994622,0.000000,1.000000,0.967700,1.000000,0.981100,1.000000,1.000000,0.994900,1.000000,1.000000
3,0.061951,0.020719,0.992147,0.992147,0.992147,0.996603,0.000000,1.000000,0.973900,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,0.032557,0.032699,0.979194,0.985602,0.982387,0.989810,0.000000,0.974400,0.957900,1.000000,1.000000,1.000000,0.983200,0.993200,1.000000,1.000000
5,0.002844,0.037573,0.985696,0.992147,0.988911,0.995471,0.000000,0.975000,0.977300,1.000000,1.000000,1.000000,1.000000,0.994900,1.000000,1.000000
6,0.028055,0.014497,0.988251,0.990838,0.989542,0.997170,0.000000,0.974400,0.970900,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.500000
7,0.012035,0.021144,0.990850,0.992147,0.991498,0.996886,0.000000,1.000000,0.980500,1.000000,1.000000,1.000000,1.000000,0.994900,1.000000,1.000000
8,0.045145,0.032421,0.986945,0.989529,0.988235,0.995754,0.000000,1.000000,0.970900,1.000000,0.980400,1.000000,1.000000,0.994900,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1200, training_loss=0.2295192948251497, metrics={'train_runtime': 279.3447, 'train_samples_per_second': 257.746, 'train_steps_per_second': 16.109, 'total_flos': 214489831595760.0, 'train_loss': 0.2295192948251497, 'epoch': 8.0})

## 5. Evaluation & Save

### 5.1 Final Metrics

In [17]:
metrics = trainer.evaluate()

print(f'Experiment     : {EXPERIMENT_NAME}')
print(f'Samples        : {len(sentences)}')
print(f'─' * 40)
print(f'Eval F1        : {metrics["eval_f1"]:.4f}')
print(f'Eval Precision : {metrics["eval_precision"]:.4f}')
print(f'Eval Recall    : {metrics["eval_recall"]:.4f}')
print(f'Eval Accuracy  : {metrics["eval_token_accuracy"]:.4f}')

Experiment     : indobert-bible-ner-v4
Samples        : 3000
────────────────────────────────────────
Eval F1        : 0.9921
Eval Precision : 0.9921
Eval Recall    : 0.9921
Eval Accuracy  : 0.9966


In [18]:
pattern_metrics = {
    k.replace('eval_f1_', ''): v
    for k, v in metrics.items()
    if k.startswith('eval_f1_')
}

if pattern_metrics:
    # Order by F1 ascending so weakest patterns surface first
    all_patterns  = sorted(set(eval_patterns))
    scored = [(p, pattern_metrics.get(p)) for p in all_patterns]
    scored_only = sorted([(p, f1) for p, f1 in scored if f1 is not None], key=lambda x: x[1])
    skipped = [(p, None) for p, f1 in scored if f1 is None]

    print(f'\n{"Pattern":<22}  {"F1":>6}  {"N eval":>7}')
    print(f'─' * 40)
    for pat, f1 in scored_only + skipped:
        n = sum(1 for p in eval_patterns if p == pat)
        if f1 is None:
            print(f'  {pat:<20}  {"—":>6}  {n:>6}  (skipped — too few samples)')
        else:
            bar = '█' * int(f1 * 20)
            print(f'  {pat:<20}  {f1:>6.4f}  {n:>6}  {bar}')


Pattern                     F1   N eval
────────────────────────────────────────
  none                  0.0000      21  
  noisy                 0.9739     136  ███████████████████
  book_only             1.0000      26  ████████████████████
  chapter_range         1.0000     148  ████████████████████
  cross_book            1.0000      89  ████████████████████
  cross_chapter_verse   1.0000       3  ████████████████████
  multi                 1.0000     128  ████████████████████
  single_chapter        1.0000      39  ████████████████████
  single_verse          1.0000       2  ████████████████████
  verse_range           1.0000       8  ████████████████████


In [19]:
# ── Get raw predictions on eval set ─────────────────────────────────────────
predictions_output = trainer.predict(tokenized_dataset["eval"])
raw_logits = predictions_output.predictions
pred_ids = np.argmax(raw_logits, axis=-1)
gold_ids = predictions_output.label_ids  # -100 for subword continuations

# Decode back to label strings, skipping -100
true_seqs, pred_seqs = [], []
for pred_seq, gold_seq in zip(pred_ids, gold_ids):
    true_row, pred_row = [], []
    for p, g in zip(pred_seq, gold_seq):
        if g == -100:
            continue
        true_row.append(id2label[g])
        pred_row.append(id2label[p])
    true_seqs.append(true_row)
    pred_seqs.append(pred_row)

In [20]:
from collections import defaultdict

# ── Token-level confusion per pattern ───────────────────────────────────────
# error_buckets[pattern] = list of (gold_label, pred_label) pairs
error_buckets = defaultdict(list)

for true_row, pred_row, pattern in zip(true_seqs, pred_seqs, eval_patterns):
    for g, p in zip(true_row, pred_row):
        error_buckets[pattern].append((g, p))

# ── Summary table ────────────────────────────────────────────────────────────
print(f'{"Pattern":<22}  {"Tokens":>6}  {"Correct":>7}  {"Acc":>5}  {"FP":>4}  {"FN":>4}  {"Wrong tag":>9}')
print('─' * 70)

error_records = []

for pat in sorted(set(eval_patterns)):
    pairs = error_buckets[pat]
    total  = len(pairs)
    correct = sum(1 for g, p in pairs if g == p)

    # False Positive: model says entity, gold says O
    fp = sum(1 for g, p in pairs if g == 'O' and p != 'O')
    # False Negative: gold says entity, model says O
    fn = sum(1 for g, p in pairs if g != 'O' and p == 'O')
    # Wrong tag: both non-O but disagree (B vs I confusion)
    wrong_tag = sum(1 for g, p in pairs if g != 'O' and p != 'O' and g != p)

    acc = correct / total if total else 0
    print(f'  {pat:<20}  {total:>6}  {correct:>7}  {acc:>5.3f}  {fp:>4}  {fn:>4}  {wrong_tag:>9}')

    error_records.append({
        'pattern': pat, 'total': total, 'correct': correct,
        'acc': acc, 'fp': fp, 'fn': fn, 'wrong_tag': wrong_tag,
    })

Pattern                 Tokens  Correct    Acc    FP    FN  Wrong tag
──────────────────────────────────────────────────────────────────────
  book_only                 56       56  1.000     0     0          0
  chapter_range            541      541  1.000     0     0          0
  cross_book               486      486  1.000     0     0          0
  cross_chapter_verse        9        9  1.000     0     0          0
  multi                   1270     1270  1.000     0     0          0
  noisy                    844      839  0.994     1     2          2
  none                     122      115  0.943     7     0          0
  single_chapter           161      161  1.000     0     0          0
  single_verse              14       14  1.000     0     0          0
  verse_range               30       30  1.000     0     0          0


In [21]:
# ── Show the worst sentences per pattern ────────────────────────────────────
# Reconstruct token list from eval split (stored before tokenization removed it)
eval_tokens = [eval_sent[i] for i in range(len(eval_sent))]

print('\n── Failing sentences by pattern ────────────────────────────────────────\n')

for pat in sorted(set(eval_patterns)):
    failures = []
    for i, (true_row, pred_row, pattern) in enumerate(zip(true_seqs, pred_seqs, eval_patterns)):
        if pattern != pat:
            continue
        if true_row == pred_row:
            continue  # correct, skip

        tokens = eval_tokens[i]
        # Align tokens to non-subword positions (same length as true_row)
        annotated = []
        for tok, g, p in zip(tokens, true_row, pred_row):
            mark = '' if g == p else f'[gold={g} pred={p}]'
            annotated.append(f'{tok}{mark}')

        failures.append(' '.join(annotated))

    if not failures:
        print(f'[{pat}] ✓ no errors\n')
    else:
        print(f'[{pat}] {len(failures)} failing sentences:')
        for sent in failures[:5]:   # cap at 5 per pattern
            print(f'  → {sent}')
        if len(failures) > 5:
            print(f'  ... and {len(failures) - 5} more')


── Failing sentences by pattern ────────────────────────────────────────

[book_only] ✓ no errors

[chapter_range] ✓ no errors

[cross_book] ✓ no errors

[cross_chapter_verse] ✓ no errors

[multi] ✓ no errors

[noisy] 4 failing sentences:
  → Jojo Dan[gold=B-BIBLE_REF pred=O] 2 - Zefanya 3 done🙏
  → 32. Sri Soejanto done Roma 16 - 1[gold=B-BIBLE_REF pred=I-BIBLE_REF] Korintus 1 - 2 👉🎄
  → 3 Yoh 1 ✅ Yudas[gold=B-BIBLE_REF pred=O] ✅ Wah 1-2 ✅
  → 1 Taw 29 dan[gold=O pred=I-BIBLE_REF] 2[gold=B-BIBLE_REF pred=I-BIBLE_REF] Taw 1 selesai 🙏
[none] 4 failing sentences:
  → 1-2-3[gold=O pred=B-BIBLE_REF] gas
  → Jam[gold=O pred=B-BIBLE_REF] 7-9[gold=O pred=I-BIBLE_REF] kita meeting
  → buka[gold=O pred=B-BIBLE_REF] hal[gold=O pred=I-BIBLE_REF] 141[gold=O pred=I-BIBLE_REF]
  → Markus[gold=O pred=B-BIBLE_REF] dan Lukas lagi meeting
[single_chapter] ✓ no errors

[single_verse] ✓ no errors

[verse_range] ✓ no errors



### 5.2 Save Model

In [22]:
SAVE_PATH.mkdir(parents=True, exist_ok=True)
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v4
